# Compartment calling in *M.racemosus*

In this file we do compartment calling for *M. racemosus* using `cis_eigs` from cooltools. GC content is used as a phasing track, meaning eigenvectors are oriented based on this. 

## Import packages 

In [ ]:
# Standard packages
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import os, subprocess

In [ ]:
# Packages for handeling cooler files 
import cooler
import cooltools
import bioframe
import cooltools.lib.plotting

In [ ]:
# Packages for plotting the heatmap
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.4'):
    raise AssertionError("tutorials rely on cooltools version 0.5.4 or higher,"+
                         "please check your cooltools version and update to the latest")

## Importing files which includes the genome file, mcool file and gene file

Import the files we needd. Import the subgenome 2  file using `load_fasta` from the bioframe package, which we use to find the GC content for each bin. Files containing the gene density and repeat density for each bin is also imported.

Used `list_coolers` to find out which resolutions that were available in the cooler file. Then imported the mcool file using 80 000 resolution. The tutorial on cooltools used 100 000, but the closest we had was 80 000, therefore we used that. Found out that this was the resolution showing best "quality". 

In [ ]:
# Import the genome which can be used to calculate the GC content
# Load the fasta genome
MucRace_genome = bioframe.load_fasta('/Users/emma/Documents/NMBU/Master/Master/gzMucRace1/gzMucRace1/gzMucRace1_genome.fasta') 

In [ ]:
# Load BED file
gene_density_MucRace = pd.read_csv('/Users/emma/Documents/NMBU/Master/Master/gzMucRace1/gzMucRace1/gene_density_80000.bed', 
                        sep='\t', header=None, names=['chrom','start','end', 'gene_count'])

In [ ]:
# Load BED file
repeat_density_MucRace = pd.read_csv('/Users/emma/Documents/NMBU/Master/Master/gzMucRace1/Repeats/repeats_density_MucRace_80000.bed', 
                        sep='\t', header=None, names=['chrom','start','end', 'repeat_count'])

In [ ]:
# To print which resolutions are stored in the mcool, use list_coolers
cooler.fileops.list_coolers('/Users/emma/Documents/NMBU/Master/Master/gzMucRace1/nfcore/Contact maps/gzMucRace1.mcool')

In [ ]:
# Load the data at resolution 80 000
MucRace_file = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/gzMucRace1/nfcore/Contact maps/gzMucRace1.mcool::resolutions/80000') 
resolution = MucRace_file.binsize
print(resolution)

## Calculating per-chromosome compartmentalization using cooltools

After importing the file with chosen resolution we used the bioframe package we get the genomic bins from the cooler object using `bins` which gives us the bin coordinates of the Hi-C matrix, meaning chromosome, start and end of bin. `[:]` means that it gets converted into a pandas dataframe. Then we read the *M.racemosus* genome using `bioframe.load_fasta`, and this returnes a dictonary-like object mapping chromosome names to sequences. For example, MucRace'subgenome2_scaffold_2] gives the sequence of chromosome 2 in subgenome 2. Then the GC content for each bin is calculated using `frac_gc`, resulting in a dataframe with chrom, start, end and fraction of GC (number between 0 and 1) for each bin. These results are saved in a tsv file using the `to_csv` function. 

In [ ]:
# get the bins from the cooler file
bins = MucRace_file.bins()[:] 
display(bins)

In Cooltools, a view defines specific genomic regions for analysis, and here a simple view is created that includes all chromosomes in the cooler, allowing downstream analyses like eigendecomposition to focus only on these regions. This code makes a table that only consist of the chromosomes that are present in our cooler file, including the name of the chromosomes, end position and name which is the same as chrom. Define which chromosomes we want to use later.

In [ ]:
# Define the genomic regions used for analysis 
view_df = pd.DataFrame({'chrom': MucRace_file.chromnames,
                        'start': 0,
                        'end': MucRace_file.chromsizes.values,
                        'name': MucRace_file.chromnames}
                      )
display(view_df)

To capture the pattern of compartmentalization within-chromosomes, in cis (on the same chromsome), cooltools `eigs_cis` first removes the dependence of contact frequency by distance, and then performs eigenedecompostion. `eigs_cis` performes the eigen value decomposition on the cooler matrix to calculate compartment signal by finding the eigenvector that correlates best with the phasing track. It needs the arguments, cool path, track path, view and n_eigs. The track path is to a bedgraph-like file which stores the phasing track as a track-named column, and the bedgraph-like format assumes tab-separated columns including chrom, start, stop and track-name (ex. gc). The view argument is a path to a BED file which defines which regions of the chromosome to use. In this case we listed the chromosomes in the contact matrix in the code above, and we use that here. The n_eigs argument is the number of eigenvectors that we should compute, here this is three. eigs-cis finds which of the eigenvectors that correlates best either positivly or negativly with the phasing track, and if the correlation is negative it flips the eigenvalues to match the track orientation. There are other similar functions like eigs-trans, expected-cis and expected-trans (Abdennur,2024). 

Here we use the GC track we just made above, the cooler file and the view dataframe. The function `view_d` tells `eigs_cis` that if this function is provided, eigenvectors should only be calculated for the regions of the view only (the dataframe we made above), otherwise chromosome-wide eigenvectors are computed, for chromosomes specified in phasing track. `n_eigs` tells the code how many eigenvectors we want, in this case three.

In [ ]:
# obtain first 3 eigenvectors
cis_eigs = cooltools.eigs_cis(
                        MucRace_file,
                        gene_density_MucRace,
                        view_df=view_df, 
                        n_eigs=3,
                        )

# cis_eigs[0] returns eigenvalues, here we focus on eigenvectors
eigenvector_track_all = cis_eigs[1][['chrom','start','end','E1', 'E2', 'E3']] # I added E2 and E3 because I wanted to see all the eigenvectors
display(eigenvector_track_all.head())

eigenvector_track = cis_eigs[1][['chrom','start','end','E1']] # the code from cooltools
display(eigenvector_track.head())

In [ ]:
import numpy as np
import pandas as pd

# assuming eigenvector_track_all has 'chrom', 'start', 'end', 'E1'
# and gc_track is a dataframe with 'chrom', 'start', 'end', 'GC'

# Merge by chrom and start/end (or nearest bin)
merged = pd.merge(eigenvector_track_all, gene_density_MucRace, on=['chrom','start','end'])
# Drop rows with any NaN values in E1 or gene_count
merged_clean = merged.dropna(subset=['E1','E2', 'E3', 'gene_count'])

correlation = np.corrcoef(merged_clean['E1'], merged_clean['gene_count'])[0,1]
print(f"Correlation between E1 and gene density: {correlation:.2f}")

In [ ]:
# Save the eigenvectors in a tsv file
eigenvector_track_all.to_csv("eigenvectors_MucRace80kb.tsv", sep='\t', index=False)
# Used this for generating the gene density boxplots 


### Import centromeres predicted using *M.lusitanicus* as a reference

In [ ]:
centromeres = {
    "subgenome1_scaffold_2": (4713049, 5677333),
    "subgenome1_scaffold_3": (2619479, 16444124)
}

## Plotting compartments calling

In [ ]:
# Choosen chromosome 
chrom = 'subgenome1_scaffold_3'

# Shorten the chromosome name, so we get only the chromosome number
chromosome_number = chrom.split('_')[-1]  
print(chromosome_number)  

# Finds centromere positions in the centromere list
centromere_start, centromere_end = centromeres[chrom]

# Convert centromere positions to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution

# Extract Hi-C matrix for this chromosome
matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]

# Extract eigenvector for this chromosome
evec_sub1_1 = eigenvector_track_all.query(f'chrom == "{chrom}"')['E1'].values

# Extract gene-density vector for this chromosome
gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values
print(len(gene_density), n_bins)

# Extract repeat-density vector for this chromosome
repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values
print(len(repeat_density), n_bins)


# Plot Hi-C matrix
# Hi-C matrix is named ax
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.matshow(matrix, norm=LogNorm(vmax=0.1), cmap='bwr')
ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

# Colorbar for Hi-C matrix indicating contact frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax, label='Corrected frequencies')
cbar.ax.yaxis.label.set_size(12)  # Change fontsize for "corrected frequencies"


# y-axis labels  display genomic coordinates in Mb 
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6  # Convert to Mb
ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{label:.2f}" for label in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize = 12)
#ax.set_xlabel("A.rouxii Sub1_1")

# Add Centromeres as lines
ax.axhline(centromere_start_bin, color='green', linestyle=':', label='Centromere')
ax.axvline(centromere_start_bin, color='green', linestyle=':')
ax.axhline(centromere_end_bin, color='green', linestyle=':')
ax.axvline(centromere_end_bin, color='green', linestyle=':')
ax.legend(loc='upper right')

# Adds a new axis on top of the other plot with Eigenvector value (E1) this is called ax1
ax1 = divider.append_axes("top", size="20%", pad=0.25, sharex=ax)
weights = MucRace_file.bins()[:]['weight'].values
ax1.plot([0, 500], [0, 0], 'k', lw=0.25)
ax1.plot(evec_sub1_1, label='E1')

ax1.set_ylabel('E1', fontsize = 12)
ax1.set_xticks([])

# Add gene density track above E1 track, called ax2
ax2 = divider.append_axes("top", size="20%", pad=0.15, sharex=ax1)
ax2.plot(gene_density, lw=1, color='red', linestyle='-', alpha=0.8)
ax2.set_ylabel("Gene\nDensity", fontsize = 12)
ax2.set_xticks([])

ax2.set_ylim(0, max(gene_density) * 1.1) # Nicer y-axis limits

# Add repeat density track above E1 track, called ax2
ax3 = divider.append_axes("top", size="20%", pad=0.15, sharex=ax2)
ax3.plot(repeat_density, lw=1, color='purple', linestyle='-', alpha=0.8)
ax3.set_ylabel("Repeat\nDensity", fontsize = 12)
ax3.set_xticks([])

ax3.set_ylim(0, max(repeat_density) * 1.1) # Nicer y-axis limits

# Overlay compartment boundaries
boundaries = np.where(np.diff((evec_sub1_1 > 0).astype(int)))[0]
for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)

fig.text(
    0.5, 0.91,
    f"Chromosome {chromosome_number} - M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#B25D91',
    fontname='Times New Roman'
)

# Show the plot
plt.show()

Largest chromosome

In [ ]:
# Choosen chromosome 
chrom = 'subgenome1_scaffold_1'

# Shorten the chromosome name, so we get only the chromosome number
chromosome_number = chrom.split('_')[-1]  
print(chromosome_number)  

# Finds centromere positions in the centromere list
#centromere_start, centromere_end = centromeres[chrom]

# Convert centromere positions to bins
#centromere_start_bin = centromere_start // resolution
#centromere_end_bin = centromere_end // resolution

# Extract Hi-C matrix for this chromosome
matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]

# Extract eigenvector for this chromosome
evec_sub1_1 = eigenvector_track_all.query(f'chrom == "{chrom}"')['E1'].values

# Extract gene-density vector for this chromosome
gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values
print(len(gene_density), n_bins)

# Extract repeat-density vector for this chromosome
repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values
print(len(repeat_density), n_bins)


# Plot Hi-C matrix
# Hi-C matrix is named ax
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.matshow(matrix, norm=LogNorm(vmax=0.1), cmap='fall')
ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

# Colorbar for Hi-C matrix indicating contact frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax, label='Corrected frequencies')
cbar.ax.yaxis.label.set_size(12)  # Change fontsize for "corrected frequencies"


# y-axis labels  display genomic coordinates in Mb 
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6  # Convert to Mb
ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{label:.2f}" for label in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize = 12)

# Add Centromeres as lines
#ax.axhline(centromere_start_bin, color='green', linestyle=':', label='Centromere')
#ax.axvline(centromere_start_bin, color='green', linestyle=':')
#ax.axhline(centromere_end_bin, color='green', linestyle=':')
#x.axvline(centromere_end_bin, color='green', linestyle=':')
#ax.legend(loc='upper right')

# Adds a new axis on top of the other plot with Eigenvector value (E1) this is called ax1
ax1 = divider.append_axes("top", size="20%", pad=0.25, sharex=ax)
weights = MucRace_file.bins()[:]['weight'].values
ax1.plot([0, 500], [0, 0], 'k', lw=0.25)
ax1.plot(evec_sub1_1, label='E1')

ax1.set_ylabel('E1', fontsize = 12)
ax1.set_xticks([])

# Add gene density track above E1 track, called ax2
ax2 = divider.append_axes("top", size="20%", pad=0.15, sharex=ax1)
ax2.plot(gene_density, lw=1, color='red', linestyle='-', alpha=0.8)
ax2.set_ylabel("Gene\nDensity", fontsize = 12)
ax2.set_xticks([])

ax2.set_ylim(0, max(gene_density) * 1.1) # Nicer y-axis limits

# Add repeat density track above E1 track, called ax2
ax3 = divider.append_axes("top", size="20%", pad=0.15, sharex=ax2)
ax3.plot(repeat_density, lw=1, color='purple', linestyle='-', alpha=0.8)
ax3.set_ylabel("Repeat\nDensity", fontsize = 12)
ax3.set_xticks([])

ax3.set_ylim(0, max(repeat_density) * 1.1) # Nicer y-axis limits

# Overlay compartment boundaries
boundaries = np.where(np.diff((evec_sub1_1 > 0).astype(int)))[0]
for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)

fig.text(
    0.5, 0.91,
    f"Chromosome {chromosome_number} - M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#B25D91',
    fontname='Times New Roman'
)

# Show the plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ===============================
# Choose chromosome
# ===============================

chrom = 'subgenome1_scaffold_2'

chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# ===============================
# Centromere coordinates
# ===============================

centromere_start, centromere_end = centromeres[chrom]

centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution

# ===============================
# Hi-C matrix
# ===============================

matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]

# ===============================
# Eigenvectors for chromosome
# ===============================

evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')

E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# ===============================
# Gene density
# ===============================

gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values

# ===============================
# Repeat density
# ===============================

repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# ===============================
# Plot Hi-C
# ===============================

fig, ax = plt.subplots(figsize=(10, 10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1),
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

# ===============================
# Colorbar
# ===============================

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)

cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# ===============================
# Genomic y-axis (Mb)
# ===============================

genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=12)

# ===============================
# Centromere lines
# ===============================

ax.axhline(centromere_start_bin, color='green', linestyle=':')
ax.axvline(centromere_start_bin, color='green', linestyle=':')

ax.axhline(centromere_end_bin, color='green', linestyle=':')
ax.axvline(centromere_end_bin, color='green', linestyle=':')

ax.legend(['Centromere'], loc='upper right')

# ===============================
# E1 track
# ===============================

ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)

ax1.plot(E1, color='black', lw=1)
ax1.axhline(0, color='grey', lw=0.5)

ax1.set_ylabel('E1', fontsize=12)
ax1.set_xticks([])

# ===============================
# E2 track
# ===============================

ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)

ax2.plot(E2, color='blue', lw=1)
ax2.axhline(0, color='grey', lw=0.5)

ax2.set_ylabel('E2', fontsize=12)
ax2.set_xticks([])

# ===============================
# E3 track
# ===============================

ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax2)

ax3.plot(E3, color='green', lw=1)
ax3.axhline(0, color='grey', lw=0.5)

ax3.set_ylabel('E3', fontsize=12)
ax3.set_xticks([])

# ===============================
# Gene density
# ===============================

ax4 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax3)

ax4.plot(gene_density, lw=1, color='red', alpha=0.8)

ax4.set_ylabel("Gene\nDensity", fontsize=12)
ax4.set_xticks([])
ax4.set_ylim(0, max(gene_density) * 1.1)

# ===============================
# Repeat density
# ===============================

ax5 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax4)

ax5.plot(repeat_density, lw=1, color='purple', alpha=0.8)

ax5.set_ylabel("Repeat\nDensity", fontsize=12)
ax5.set_xticks([])
ax5.set_ylim(0, max(repeat_density) * 1.1)

# ===============================
# Compartment boundaries from E1
# ===============================

boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)

# ===============================
# Title
# ===============================

fig.text(
    0.5, 0.91,
    f"Chromosome {chromosome_number} – M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#B25D91',
    fontname='Times New Roman'
)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ===============================
# Choose chromosome
# ===============================

chrom = 'subgenome1_scaffold_2'

chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# ===============================
# Centromere coordinates
# ===============================

centromere_start, centromere_end = centromeres[chrom]

centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution

# ===============================
# Hi-C matrix
# ===============================

matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]

# ===============================
# Eigenvectors for chromosome
# ===============================

evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')

E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# ===============================
# Gene density
# ===============================

gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values

# ===============================
# Repeat density
# ===============================

repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# ===============================
# Plot Hi-C
# ===============================

fig, ax = plt.subplots(figsize=(10, 10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1),
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

# ===============================
# Colorbar
# ===============================

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)

cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# ===============================
# Genomic y-axis (Mb)
# ===============================

genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=12)

# ===============================
# Centromere lines
# ===============================

ax.axhline(centromere_start_bin, color='green', linestyle=':')
ax.axvline(centromere_start_bin, color='green', linestyle=':')

ax.axhline(centromere_end_bin, color='green', linestyle=':')
ax.axvline(centromere_end_bin, color='green', linestyle=':')

ax.legend(['Centromere'], loc='upper right')

# ===============================
# E1 track
# ===============================

ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)

ax1.plot(E1, color='#1f77b4', lw=1)
ax1.axhline(0, color='grey', lw=0.5)

ax1.set_ylabel('E1', fontsize=12)
ax1.set_xticks([])

# ===============================
# E2 track
# ===============================

ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)

ax2.plot(E2, color='#4DB6AC', lw=1)
ax2.axhline(0, color='grey', lw=0.5)

ax2.set_ylabel('E2', fontsize=12)
ax2.set_xticks([])

# ===============================
# E3 track
# ===============================

ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax2)

ax3.plot(E3, color='#9B5DE5', lw=1)
ax3.axhline(0, color='grey', lw=0.5)

ax3.set_ylabel('E3', fontsize=12)
ax3.set_xticks([])

# ===============================
# Gene density
# ===============================

ax4 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax3)

ax4.plot(gene_density, lw=1, color='#d62728', alpha=0.8)

ax4.set_ylabel("Gene\nDensity", fontsize=12)
ax4.set_xticks([])
ax4.set_ylim(0, max(gene_density) * 1.1)

# ===============================
# Repeat density
# ===============================

ax5 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax4)

ax5.plot(repeat_density, lw=1, color='#ff7f0e', alpha=0.8)

ax5.set_ylabel("Repeat\nDensity", fontsize=12)
ax5.set_xticks([])
ax5.set_ylim(0, max(repeat_density) * 1.1)

# ===============================
# Compartment boundaries from E1
# ===============================

boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)

# ===============================
# Title
# ===============================

fig.text(
    0.5, 0.91,
    f"Chromosome {chromosome_number} – M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#B25D91',
    fontname='Times New Roman'
)

plt.show()


In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ===============================
# Choose chromosome
# ===============================

chrom = 'subgenome1_scaffold_2'

chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# ===============================
# Centromere coordinates
# ===============================

centromere_start, centromere_end = centromeres[chrom]

centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution

# ===============================
# Hi-C matrix
# ===============================

matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]

# ===============================
# Eigenvectors for chromosome
# ===============================

evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')

E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# ===============================
# Gene density
# ===============================

gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values

# ===============================
# Repeat density
# ===============================

repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# ===============================
# Plot Hi-C
# ===============================

fig, ax = plt.subplots(figsize=(10, 10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1),
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

# ===============================
# Colorbar
# ===============================
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)

cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

# Genomic axis
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)


# Centromere lines
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)

ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)

# ===============================
# E1 track
# ===============================

# E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
# Color compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=10)
ax1.set_xticks([])


# E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
# Color compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=10)
ax2.set_xticks([])

# E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
# Color compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=10)
ax3.set_xticks([])


# Gene density
ax4 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax3)

ax4.plot(gene_density, lw=1, color='green', alpha=0.8)

ax4.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax4.set_xticks([])
ax4.set_ylim(0, max(gene_density) * 1.1)


# Repeat density
ax5 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax4)

ax5.plot(repeat_density, lw=1, color='purple', alpha=0.8)

ax5.set_ylabel("Repeat\nDensity", fontsize=14)
ax5.set_xticks([])
ax5.set_ylim(0, max(repeat_density) * 1.1)


# Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


# ===============================
# Title
# ===============================

fig.text(
    0.5, 0.91,
    f"Chromosome {chromosome_number} – M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#B25D91'
)

plt.show()


# For results

In [ ]:
# Set fontsizes for result plots (to be the same across species)
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 35,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
})

In [ ]:
# Largest chromosome with annotated centromere
chrom = 'subgenome1_scaffold_2'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
centromere_start, centromere_end = centromeres[chrom]
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = MucRace_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_MucRace[gene_density_MucRace["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_MucRace[repeat_density_MucRace["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,12))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)')
cbar.ax.tick_params()

## Genomic x-axis in Mb
### Default gave a lot of ticks 
n_ticks = 8
genomic_ticks = np.linspace(0, n_bins-1, n_ticks, dtype=int)

genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.1f}" for x in genomic_labels])

ax.set_ylabel("Genomic Position (Mb)")

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1.25)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1.25)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1.25)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1.25)
#ax.legend(['Centromere'], loc='upper right')


## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', labelpad=10)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', labelpad=5)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', labelpad=5)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='black', alpha=0.9)
ax_gene.set_ylabel("Gene\nDensity",labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='black', alpha=0.9)
ax_repeat.set_ylabel("Repeat\nDensity")
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Align all y-axis labels
for a in [ax1, ax2, ax3, ax_gene, ax_repeat]:
    a.yaxis.set_label_coords(-0.11, 0.5)


# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – M. racemosus ({resolution//1000} kb)",
    ha="center",
    va="top",
    color='#B25D91',
    fontsize=25,
    fontname='Times New Roman'
)

plt.show()
